# Demo: Unfolding with uncertainty propagation

This notebook demonstrates a classic inverse problem: you observe a smeared (detector-level) spectrum $y$ and want to estimate the true spectrum $x$.

We use the linear model $y = R x$ (plus Poisson counting noise) and compare:
- **Tikhonov unfolding** (ridge-style regularization): $(R^T W R + ambda^2 L^T L) x = R^T W y$
- **Truncated SVD** unfolding: invert only the first $k$ singular modes

We scan regularization strength and validate with toy Monte Carlo: stability, bias–variance tradeoff, and pull widths.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from unfolding.response import build_response_matrix, smear_truth
from unfolding.tikhonov import tikhonov_unfold
from unfolding.tsvd import tsvd_unfold
from unfolding.toys import run_toy_mc

plt.rcParams.update({"figure.dpi": 120})

In [ ]:
# Problem setup
rng = np.random.default_rng(123)
n_truth = 25
n_reco = 25

# A smooth-ish truth with structure
bins = np.arange(n_truth)
truth = (
    1200 * np.exp(-0.12 * bins)
    + 140 * np.exp(-0.5 * ((bins - 10) / 2.0) ** 2)
    + 90 * np.exp(-0.5 * ((bins - 18) / 1.7) ** 2)
).astype(float)

R = build_response_matrix(n_truth=n_truth, n_reco=n_reco, sigma=1.8, seed=7)
y_obs = smear_truth(truth, R, rng=rng, poisson=True)

# Covariance of y under Poisson approximation
cov_y = np.diag(np.maximum(y_obs, 1.0))

truth.sum(), y_obs.sum()

In [ ]:
# Scan Tikhonov regularization (lambda)
lambdas = np.logspace(-2.5, 1.0, 22)

xs = []
sigmas = []
residual_norm = []
solution_norm = []

for lam in lambdas:
    x_hat, cov_x = tikhonov_unfold(y_obs, R, reg=float(lam), cov_y=cov_y, diff_order=2, nonneg=False)
    xs.append(x_hat)
    sigmas.append(np.sqrt(np.maximum(np.diag(cov_x), 0.0)))
    residual_norm.append(np.linalg.norm(R @ x_hat - y_obs))
    solution_norm.append(np.linalg.norm(x_hat))

xs = np.asarray(xs)
sigmas = np.asarray(sigmas)
residual_norm = np.asarray(residual_norm)
solution_norm = np.asarray(solution_norm)

# Pick a reasonable lambda by a crude L-curve heuristic: maximize curvature in log-log space
t = np.log(lambdas)
x1 = np.log(residual_norm)
y1 = np.log(solution_norm)

dx = np.gradient(x1, t)
dy = np.gradient(y1, t)
ddx = np.gradient(dx, t)
ddy = np.gradient(dy, t)
curv = np.abs(dx * ddy - dy * ddx) / np.maximum((dx * dx + dy * dy) ** 1.5, 1e-12)
i_best = int(np.argmax(curv))
lam_best = float(lambdas[i_best])
lam_best

In [ ]:
# Plot truth, observation, and a few unfolded solutions
fig, ax = plt.subplots(figsize=(8, 4))
ax.step(bins, truth, where='mid', label='truth', lw=2)
ax.step(bins, y_obs, where='mid', label='observed reco (counts)', lw=1.5, alpha=0.8)

for idx in [0, i_best, -1]:
    ax.step(bins, xs[idx], where='mid', label=f'unfolded (λ={lambdas[idx]:.2g})')

ax.set_xlabel('bin')
ax.set_ylabel('counts')
ax.set_title('Unfolding: effect of regularization')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Bias–variance (from linearized covariance) for the chosen lambda
x_best = xs[i_best]
x_best_sigma = sigmas[i_best]

fig, ax = plt.subplots(figsize=(8, 4))
ax.step(bins, x_best - truth, where='mid', label='bias (x̂ - truth)')
ax.step(bins, x_best_sigma, where='mid', label='σ (from propagated cov)', alpha=0.8)
ax.axhline(0.0, color='k', lw=1)
ax.set_xlabel('bin')
ax.set_title(f'Chosen λ≈{lam_best:.2g}: bias vs propagated uncertainty')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Toy MC validation: pull widths should be ~1 when covariance is realistic
def unfold_best(y):
    cov_y_toy = np.diag(np.maximum(y, 1.0))
    return tikhonov_unfold(y, R, reg=lam_best, cov_y=cov_y_toy, diff_order=2, nonneg=False)

toy = run_toy_mc(truth, R, n_toys=400, unfold=unfold_best, rng=np.random.default_rng(999))

fig, ax = plt.subplots(figsize=(8, 4))
ax.step(bins, toy['pull_mean'], where='mid', label='pull mean')
ax.step(bins, toy['pull_std'], where='mid', label='pull std')
ax.axhline(0.0, color='k', lw=1)
ax.axhline(1.0, color='k', lw=1, ls='--')
ax.set_ylim(-0.5, 2.0)
ax.set_xlabel('bin')
ax.set_title('Toy MC validation (Tikhonov, chosen λ)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Optional: compare truncated SVD with varying k
ks = list(range(3, min(n_truth, n_reco) + 1, 3))
xs_k = []
sig_k = []

for k in ks:
    x_hat, cov_x = tsvd_unfold(y_obs, R, k=k, cov_y=cov_y)
    xs_k.append(x_hat)
    sig_k.append(np.sqrt(np.maximum(np.diag(cov_x), 0.0)))

xs_k = np.asarray(xs_k)
sig_k = np.asarray(sig_k)

fig, ax = plt.subplots(figsize=(8, 4))
ax.step(bins, truth, where='mid', label='truth', lw=2)
for idx, k in enumerate(ks[:4]):
    ax.step(bins, xs_k[idx], where='mid', label=f'TSVD k={k}', alpha=0.9)
ax.set_xlabel('bin')
ax.set_title('TSVD unfolding (first few k)')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Save key figures for the LaTeX note
from pathlib import Path

outdir = Path('..') / 'docs' / 'figures'
outdir.mkdir(parents=True, exist_ok=True)

# Figure 1: regularization scan (L-curve)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(residual_norm, solution_norm, marker='o', ms=3)
ax.plot(residual_norm[i_best], solution_norm[i_best], marker='o', ms=7)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('||R x̂ - y||')
ax.set_ylabel('||x̂||')
ax.set_title('L-curve (Tikhonov)')
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
fig.savefig(outdir / 'lcurve.png')
plt.close(fig)

# Figure 2: chosen solution vs truth
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.step(bins, truth, where='mid', label='truth', lw=2)
ax.step(bins, y_obs, where='mid', label='observed reco', lw=1.5, alpha=0.8)
ax.step(bins, x_best, where='mid', label=f'unfolded (λ≈{lam_best:.2g})')
ax.set_xlabel('bin')
ax.set_ylabel('counts')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(outdir / 'unfolded_vs_truth.png')
plt.close(fig)

# Figure 3: pulls
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.step(bins, toy['pull_mean'], where='mid', label='pull mean')
ax.step(bins, toy['pull_std'], where='mid', label='pull std')
ax.axhline(0.0, color='k', lw=1)
ax.axhline(1.0, color='k', lw=1, ls='--')
ax.set_ylim(-0.5, 2.0)
ax.set_xlabel('bin')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(outdir / 'pulls.png')
plt.close(fig)

sorted([p.name for p in outdir.glob('*.png')])